In [1]:
# --- Step 1: Import Libraries ---
import pandas as pd
import sqlite3
import os

# --- Step 2: Create Folder Structure ---
folders = ["raw", "processed", "output"]
for folder in folders:
    if not os.path.exists(folder):
        os.makedirs(folder)


In [7]:
# --- Step 3: Extract (Load Raw Retail Sales Data) ---
# Upload your retail_sales.csv file into the "raw" folder in Colab
retail_df = pd.read_csv("raw/retail_sales_dataset.csv")

print("Raw dataset loaded successfully!")
print("Initial row count:", len(retail_df))

# --- Step 4: Transform (Clean + Process) ---
# Remove missing values
retail_df = retail_df.dropna()

# Remove duplicates
retail_df = retail_df.drop_duplicates()

# Standardize column names
retail_df.columns = retail_df.columns.str.lower().str.replace(" ", "_")

# Rename 'transaction_id' to 'order_id' and 'total_amount' to 'sales'
if 'transaction_id' in retail_df.columns:
    retail_df.rename(columns={'transaction_id': 'order_id'}, inplace=True)
if 'total_amount' in retail_df.columns:
    retail_df.rename(columns={'total_amount': 'sales'}, inplace=True)

# Convert datatypes (example: date and sales)
if "date" in retail_df.columns:
    retail_df["date"] = pd.to_datetime(retail_df["date"], errors="coerce")
if "sales" in retail_df.columns:
    retail_df["sales"] = retail_df["sales"].astype(float)

# Create derived columns (removed margin and segment as 'cost' is not available)
# if "sales" in retail_df.columns and "cost" in retail_df.columns:
#     retail_df["margin"] = retail_df["sales"] - retail_df["cost"]
#     retail_df["segment"] = retail_df["sales"].apply(lambda x: "High" if x > 500 else "Low")

print("Data cleaned and transformed!")

# --- Step 5: Split into Separate Outputs ---
# Adjusted to include only available columns based on the dataset
customers = retail_df[["customer_id"]].drop_duplicates()
orders = retail_df[["order_id", "customer_id", "date", "sales"]]
products = retail_df[["product_category"]].drop_duplicates()

# --- Step 6: Load (Export to CSV + SQLite) ---
customers.to_csv("output/customers.csv", index=False)
orders.to_csv("output/orders.csv", index=False)
products.to_csv("output/products.csv", index=False)

conn = sqlite3.connect("output/database.sqlite")
customers.to_sql("customers", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
conn.close()

print("Data successfully exported to CSV and SQLite!")

# --- Step 7: Validate ---
print("Final row count (after cleaning):", len(retail_df))
print("Customers table rows:", len(customers))
print("Orders table rows:", len(orders))
print("Products table rows:", len(products))

Raw dataset loaded successfully!
Initial row count: 1000
Data cleaned and transformed!
Data successfully exported to CSV and SQLite!
Final row count (after cleaning): 1000
Customers table rows: 1000
Orders table rows: 1000
Products table rows: 3
